# 02 — SOM Training

**Feature Scaling → Self-Organizing Map → BMU Assignment**

This notebook is the second stage of the NARR SOM pipeline.  It picks up from the
Zarr store created in `01_data_preprocessing.ipynb` and produces a trained SOM whose
best-matching unit (BMU) assignments are used for analysis in notebook 03.

### Workflow
1. Load the preprocessed Zarr store
2. Rebuild and scale the feature matrix
3. Configure and train the SOM
4. Assign BMUs to all training samples
5. Produce diagnostic summary plots (node frequency, mean CAPE, mean CIN)
6. Save the trained SOM and BMU assignments to a pickle file

### Background — Self-Organizing Maps
A Self-Organizing Map (SOM) is an unsupervised neural network that projects
high-dimensional data onto a low-dimensional (typically 2-D) grid while preserving
topological relationships.  Each node on the grid learns a *prototype* (weight vector)
that represents a cluster of similar input samples.  For each training sample the
closest prototype — the **Best Matching Unit (BMU)** — is identified, and both the
BMU and its grid neighbours are nudged toward that sample.

In this application each sample is one day of gridded CAPE, CIN, and wind shear over
North America.  Similar synoptic environments cluster to the same SOM node, allowing
pattern-based analysis of severe weather potential.

### Input
- `cape_cin_NARR.zarr/` — merged NARR dataset (output of notebook 01)

### Output
- `som_output.pkl` — dictionary containing the trained `MiniSom` object, BMU array,
  and grid dimensions

## 1. Imports

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os

from sklearn.preprocessing import MinMaxScaler
from minisom import MiniSom

## 2. Configuration

All tunable parameters are collected here so they are easy to find and adjust.

| Parameter | Description |
|---|---|
| `ZARR_PATH` | Path to the Zarr store from notebook 01 |
| `OUTPUT_DIR` | Directory where `som_output.pkl` will be written |
| `SOM_X / SOM_Y` | Number of SOM columns / rows (grid size = SOM_X × SOM_Y) |
| `SIGMA` | Initial neighbourhood radius |
| `LEARNING_RATE` | Initial learning rate |
| `N_ITERATIONS` | Total training iterations (random sample draws) |
| `RANDOM_SEED` | Seed for reproducibility |

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
ZARR_PATH  = "/path/to/output/cape_cin_NARR.zarr"   # output of notebook 01
OUTPUT_DIR = "/path/to/output"
PKL_PATH   = os.path.join(OUTPUT_DIR, "som_output.pkl")

# ── SOM hyperparameters ───────────────────────────────────────────────────────
SOM_X         = 5
SOM_Y         = 5
SIGMA         = 2.0
LEARNING_RATE = 1.0
N_ITERATIONS  = 4000
RANDOM_SEED   = 0

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"SOM grid: {SOM_X} × {SOM_Y}  |  iterations: {N_ITERATIONS}")

## 3. Load Preprocessed Data

Open the Zarr store written by notebook 01.  The dataset contains daily CAPE, CIN,
and vertical wind shear on the NARR Lambert Conformal grid.

In [ ]:
ds = xr.open_zarr(ZARR_PATH)

print("Dataset loaded:")
print(ds)
print("\nVariables:", list(ds.data_vars))
print("Time range:", ds.time.values[0], "→", ds.time.values[-1])
print("n timesteps:", ds.sizes["time"])

## 4. Build and Clean the Feature Matrix

The SOM requires a 2-D array `X` of shape `(n_timesteps, n_features)`.

For each variable the spatial grid (y × x) is flattened into columns, and all
variable blocks are concatenated along the column axis.  The same cleaning pipeline
from notebook 01 is re-applied here so training can be run independently of
preprocessing.

In [ ]:
n_time = ds.sizes["time"]

X_cape = ds["cape"].values.reshape(n_time, -1).astype(np.float32)
X_cin  = ds["cin"].values.reshape(n_time, -1).astype(np.float32)

if "vwsh_kts" in ds:
    X_shr = ds["vwsh_kts"].values.reshape(n_time, -1).astype(np.float32)
    X = np.concatenate([X_cape, X_cin, X_shr], axis=1)
else:
    X = np.concatenate([X_cape, X_cin], axis=1)

print(f"Raw X shape: {X.shape}")

# Replace non-finite / fill values
X[~np.isfinite(X)] = np.nan
X[np.abs(X) > 1e20] = np.nan

# Drop all-NaN columns
good_cols = ~np.isnan(X).all(axis=0)
X = X[:, good_cols]
print(f"After all-NaN column drop: {X.shape}")

# Fill remaining NaNs with column mean
col_mean = np.nanmean(X, axis=0)
col_mean = np.nan_to_num(col_mean, nan=0.0)
nan_idx = np.where(np.isnan(X))
X[nan_idx] = col_mean[nan_idx[1]]

# Drop zero-variance columns
std = np.std(X, axis=0)
X = X[:, std > 0]

print(f"Final clean X shape: {X.shape}")
print(f"All finite: {np.isfinite(X).all()}")

## 5. Feature Scaling

The feature matrix is rescaled to [0, 1] using `MinMaxScaler` so that CAPE values
(which can reach thousands of J/kg) do not dominate CIN (typically −0 to −500 J/kg)
or wind shear in the SOM distance calculations.

> **Note:** `StandardScaler` (z-score) is an equally valid choice; MinMaxScaler is used
> here to preserve the shape of each variable's distribution.

In [ ]:
scaler = MinMaxScaler()
Xz = scaler.fit_transform(X)

print(f"Scaled matrix shape : {Xz.shape}")
print(f"Value range         : [{Xz.min():.3f}, {Xz.max():.3f}]")
print(f"NaNs in Xz          : {np.isnan(Xz).sum()}")

## 6. Train the SOM

We use the `MiniSom` library, which implements the Kohonen SOM algorithm.

Key hyperparameters:
- **`sigma`** — initial neighbourhood radius; controls how broadly the winning node's
  neighbours are updated.  Shrinks automatically over training.
- **`learning_rate`** — initial step size for weight updates.  Also decays over training.
- **`neighborhood_function`** — `'gaussian'` is standard and produces smooth topology.

Training progress is printed every 10 % of iterations.

In [ ]:
som = MiniSom(
    x=SOM_X,
    y=SOM_Y,
    input_len=Xz.shape[1],
    sigma=SIGMA,
    learning_rate=LEARNING_RATE,
    neighborhood_function="gaussian",
    random_seed=RANDOM_SEED
)

som.random_weights_init(Xz)

print(f"Training SOM ({SOM_X}×{SOM_Y}) on {Xz.shape[0]} samples × {Xz.shape[1]} features…")
som.train_random(Xz, N_ITERATIONS, verbose=True)
print("Training complete.")

## 7. Assign Best-Matching Units (BMUs)

For each training sample, find the SOM node whose weight vector is closest (Euclidean
distance) to that sample.  The result is an array `bmus` of shape `(n_timesteps, 2)`
where each row is `(col_index, row_index)` of the winning node.

In [ ]:
bmus = np.array([som.winner(x) for x in Xz])

print(f"BMU array shape: {bmus.shape}")
print("\nNode population counts (row, col):")

for j in range(SOM_Y):
    row_counts = []
    for i in range(SOM_X):
        n = int(((bmus[:, 0] == i) & (bmus[:, 1] == j)).sum())
        row_counts.append(f"{n:4d}")
    print("  row", j, ":", " ".join(row_counts))

## 8. Diagnostic Plots

Three summary heatmaps give a quick sanity check on the SOM:

- **Node frequency** — how many days map to each node; ideally no node should be
  completely empty (consider a larger grid) and no node should be overwhelmingly
  dominant (consider a smaller grid or more iterations).
- **Mean CAPE by node** — higher CAPE nodes should cluster together on the grid.
- **Mean CIN by node** — more negative (stronger) CIN should also cluster coherently.

In [ ]:
# ── Node frequency ────────────────────────────────────────────────────────────
node_counts = np.zeros((SOM_Y, SOM_X))

for j in range(SOM_Y):
    for i in range(SOM_X):
        node_counts[j, i] = ((bmus[:, 0] == i) & (bmus[:, 1] == j)).sum()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(node_counts, cmap="Blues")

for j in range(SOM_Y):
    for i in range(SOM_X):
        ax.text(i, j, int(node_counts[j, i]), ha="center", va="center", fontsize=10)

ax.set_title("Node Frequency (days per node)")
ax.set_xlabel("SOM Column")
ax.set_ylabel("SOM Row")
plt.colorbar(im, ax=ax, label="Days")
plt.tight_layout()
plt.show()

In [ ]:
# ── Mean CAPE per node ────────────────────────────────────────────────────────
node_cape = np.full((SOM_Y, SOM_X), np.nan)

for j in range(SOM_Y):
    for i in range(SOM_X):
        mask = (bmus[:, 0] == i) & (bmus[:, 1] == j)
        if mask.sum() > 0:
            node_cape[j, i] = ds["cape"].isel(time=mask).mean().values

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(node_cape, cmap="plasma")

for j in range(SOM_Y):
    for i in range(SOM_X):
        if not np.isnan(node_cape[j, i]):
            ax.text(i, j, f"{node_cape[j,i]:.0f}", ha="center", va="center", fontsize=9)

ax.set_title("Mean Domain-Averaged CAPE by SOM Node")
ax.set_xlabel("SOM Column")
ax.set_ylabel("SOM Row")
plt.colorbar(im, ax=ax, label="Mean CAPE (J/kg)")
plt.tight_layout()
plt.show()

In [ ]:
# ── Mean CIN per node ─────────────────────────────────────────────────────────
node_cin = np.full((SOM_Y, SOM_X), np.nan)

for j in range(SOM_Y):
    for i in range(SOM_X):
        mask = (bmus[:, 0] == i) & (bmus[:, 1] == j)
        if mask.sum() > 0:
            node_cin[j, i] = ds["cin"].isel(time=mask).mean().values

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(node_cin, cmap="coolwarm")

for j in range(SOM_Y):
    for i in range(SOM_X):
        if not np.isnan(node_cin[j, i]):
            ax.text(i, j, f"{node_cin[j,i]:.0f}", ha="center", va="center", fontsize=9)

ax.set_title("Mean Domain-Averaged CIN by SOM Node")
ax.set_xlabel("SOM Column")
ax.set_ylabel("SOM Row")
plt.colorbar(im, ax=ax, label="Mean CIN (J/kg)")
plt.tight_layout()
plt.show()

## 9. Save Trained SOM and BMUs

The trained `MiniSom` object and the BMU assignment array are pickled together so
they can be loaded in notebook 03 without re-training.

The pickle contains:

| Key | Contents |
|---|---|
| `som` | Trained `MiniSom` object |
| `bmus` | `(n_timesteps, 2)` array of BMU (col, row) indices |
| `som_x` | SOM column count |
| `som_y` | SOM row count |

In [ ]:
output = {
    "som"  : som,
    "bmus" : bmus,
    "som_x": SOM_X,
    "som_y": SOM_Y,
}

with open(PKL_PATH, "wb") as f:
    pickle.dump(output, f)

print(f"Saved trained SOM and BMUs to: {PKL_PATH}")

---
## Next Steps

The outputs from this notebook feed directly into **03_output_visualization.ipynb**:

- `cape_cin_NARR.zarr` — environmental composites per node
- `som_output.pkl` — BMU assignments for date matching and PPH analysis